# 01 · Inspect the dataset archives

Answers one question: **are my zips face crops (ready to train), or raw videos
(which need an extraction pass first)?**

This reads only each zip's *index*, so nothing is extracted and nothing large is
copied. Runtime: a couple of minutes.

It also checks the thing that silently breaks `build_manifest.py`: whether the
folder names inside your zips contain the tokens its label rules look for. If
they don't, the manifest comes out unlabelled — better to find that out now than
30 minutes into a full scan.

Run the cells top to bottom.

## 1 · Runtime check

Confirms you actually got a GPU, and that there is disk room to extract later.

In [ ]:
import shutil, subprocess

GB = 1024 ** 3

print("=" * 66)
print("GPU")
print("=" * 66)
try:
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True, timeout=30)
    print(out.stdout.strip() or "no GPU reported")
except FileNotFoundError:
    print("nvidia-smi not found  ->  Runtime > Change runtime type > T4 GPU")

print()
print("=" * 66)
print("DISK (/content = local disk. Data MUST live here to train, not Drive.)")
print("=" * 66)
total, used, free = shutil.disk_usage("/content")
print(f"total {total/GB:6.1f} GB     used {used/GB:6.1f} GB     free {free/GB:6.1f} GB")
print()
print("You need free space >= ~2x the total zip size to unzip here.")
print("FF++ (6.9 GB) + Celeb-DF (1.9 GB)  ->  want ~20 GB free.")
print(f"You have {free/GB:.1f} GB  ->", "OK" if free / GB > 20 else "TIGHT (see notes at bottom)")

## 2 · Mount Drive and find the zips

Lists every `.zip` in your Drive with its size, so you can copy the exact paths
into the next cell.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path

GB, MB = 1024 ** 3, 1024 ** 2
root = Path("/content/drive/MyDrive")
zips = sorted(root.rglob("*.zip"), key=lambda p: p.stat().st_size, reverse=True)

if not zips:
    print("No .zip found anywhere under /content/drive/MyDrive")
else:
    print(f"{len(zips)} zip(s), largest first:")
    print()
    for p in zips:
        size = p.stat().st_size
        shown = f"{size/GB:7.2f} GB" if size >= GB else f"{size/MB:7.1f} MB"
        print(f"  {shown}   {p}")

print()
print("Copy the paths you want into ARCHIVES in the next cell.")

## 3 · Point at your archives

Paste the paths from above. The names on the left are just labels for the report.

In [ ]:
ARCHIVES = {
    "ffpp":    "/content/drive/MyDrive/FF++.zip",
    "celebdf": "/content/drive/MyDrive/celebdf_faces.zip",
}

from pathlib import Path
for name, path in ARCHIVES.items():
    print(f"{'OK     ' if Path(path).exists() else 'MISSING'}  {name:10s} {path}")

## 4 · The inspector

Reads each zip's central directory — the index stored at the end of the file.
Nothing is extracted. On a 7 GB zip with many files this can take a minute,
because that index is itself large and Drive is slow to seek.

The label rules below are copied verbatim from `data/build_manifest.py`, so this
predicts exactly what that script will do with your paths.

In [ ]:
import re, zipfile, random
from collections import Counter, defaultdict
from pathlib import Path

# ---- copied verbatim from data/build_manifest.py ------------------------
IMG_EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
VID_EXT = {".mp4", ".avi", ".mov", ".mkv", ".webm", ".m4v"}

REAL_TOKENS = (
    "original", "originals", "real", "youtube", "actors", "pristine",
    "celeb-real", "celeb_real", "youtube-real", "youtube_real", "genuine",
)

FAKE_METHODS = (
    "deepfakes", "face2face", "faceswap", "neuraltextures", "faceshifter",
    "celeb-synthesis", "celeb_synthesis", "simswap", "inswapper", "blendface",
    "uniface", "e4s", "facedancer", "fsgan", "mobileswap", "danet",
    "wav2lip", "sadtalker", "mraa", "fomm", "tpsm", "styleheat",
    "sd15", "sdxl", "ddim", "ddpm", "stargan", "stylegan", "collab",
    "vqgan", "pixart", "midjourney", "heygen", "hyperreenact",
)


def infer_label(s):
    # Same rules as build_manifest.infer_label, on a lowercased path.
    for m in FAKE_METHODS:
        if m in s:
            return 1, m
    for t in REAL_TOKENS:
        if t in s:
            return 0, "real"
    if re.search(r"/(fake|manipulated|synth|generated)(/|_)", s):
        return 1, "unknown_fake"
    if re.search(r"/(real|authentic)(/|_)", s):
        return 0, "real"
    return -1, "unresolved"
# -------------------------------------------------------------------------


def inspect(name, path):
    path = Path(path)
    print("=" * 72)
    print(f"{name}    {path.name}    {path.stat().st_size / 1024**3:.2f} GB")
    print("=" * 72)
    print("reading zip index (no extraction)...", flush=True)

    with zipfile.ZipFile(path) as z:
        infos = [i for i in z.infolist() if not i.is_dir()]

    if not infos:
        print("EMPTY ARCHIVE")
        print()
        return None

    names = [i.filename.lower().replace("\\", "/") for i in infos]
    ext = Counter(Path(n).suffix for n in names)
    raw_gb = sum(i.file_size for i in infos) / 1024**3

    n_img = sum(c for e, c in ext.items() if e in IMG_EXT)
    n_vid = sum(c for e, c in ext.items() if e in VID_EXT)
    n_zip = ext.get(".zip", 0)
    total = len(infos)

    top = ", ".join(f"{e or '(none)'} x{c:,}" for e, c in ext.most_common(6))
    print()
    print(f"entries        : {total:,}")
    print(f"uncompressed   : {raw_gb:.2f} GB")
    print(f"top extensions : {top}")

    # ---------------------------------------------------------- verdict
    print()
    print("-" * 72)
    if n_zip >= total * 0.5:
        kind = "NESTED_ZIP"
        print("VERDICT: NESTED ZIPS - this archive just contains more zips.")
        print("         Unzip one level, then re-run this notebook on those.")
    elif n_img >= total * 0.8:
        kind = "IMAGES"
        print(f"VERDICT: FACE CROPS / FRAMES   ({n_img:,} images = {n_img/total:.0%})")
        print("         Ready to train. No extraction pass needed.")
    elif n_vid >= total * 0.5:
        kind = "VIDEOS"
        print(f"VERDICT: RAW VIDEOS   ({n_vid:,} videos = {n_vid/total:.0%})")
        print("         build_manifest.py scans images only, so it will find")
        print("         ZERO rows here. An MTCNN extraction pass is needed first.")
    else:
        kind = "MIXED"
        print(f"VERDICT: MIXED / UNCLEAR   ({n_img:,} images, {n_vid:,} videos)")
        print("         Read the sample paths below and decide.")
    print("-" * 72)

    # ----------------------------------------------------- sample paths
    print()
    print("sample paths (first 10, then 10 at random):")
    for n in names[:10]:
        print("   ", n)
    print("    ...")
    for n in random.Random(0).sample(names, min(10, len(names))):
        print("   ", n)

    # ------------------------------------------------------ folder depth
    print()
    print("folder depth (number of slashes in the path):")
    for d, c in sorted(Counter(n.count("/") for n in names).items()):
        print(f"    depth {d}: {c:,} files")

    # --------------------------------------------------- label inference
    media = [n for n in names if Path(n).suffix in IMG_EXT | VID_EXT]
    labels, methods = Counter(), Counter()
    unresolved = {}          # one example per distinct folder, not 12 siblings
    for n in media:
        lab, meth = infer_label(n)
        labels[lab] += 1
        methods[meth] += 1
        if lab < 0 and len(unresolved) < 12:
            unresolved.setdefault(str(Path(n).parent), n)

    print()
    print("-" * 72)
    print("LABEL INFERENCE - what build_manifest.py will make of these paths")
    print("-" * 72)
    print(f"    real (0)   : {labels.get(0, 0):,}")
    print(f"    fake (1)   : {labels.get(1, 0):,}")
    print(f"    UNRESOLVED : {labels.get(-1, 0):,}")
    print()
    print("    methods detected:",
          ", ".join(f"{m} x{c:,}" for m, c in methods.most_common(10)))

    if labels.get(-1, 0):
        print()
        print(f"    !! {labels[-1]/max(1,len(media)):.0%} of files could not be labelled.")
        print("       Add a matching token to REAL_TOKENS / FAKE_METHODS in")
        print("       data/build_manifest.py for paths like these:")
        for n in unresolved.values():
            print("        ", n)
    else:
        print()
        print("    All files labelled. build_manifest.py will work on this.")

    # -------------------------------------------------- clip grouping
    if kind == "IMAGES":
        parents = Counter(str(Path(n).parent) for n in media)
        counts = sorted(parents.values())
        median = counts[len(counts) // 2]
        print()
        print("-" * 72)
        print("CLIP GROUPING")
        print("-" * 72)
        print(f"    distinct folders  : {len(parents):,}   (~ number of videos)")
        print(f"    frames per folder : min {counts[0]}  median {median}  max {counts[-1]}")
        if median < 16:
            print(f"    !! median {median} is below DATA.clip_len = 16.")
            print("       ClipDataset will pad clips by repeating frames.")
            print("       Lower clip_len instead.")
        else:
            print("    Enough frames per video for clip_len = 16.")

    print()
    return {"kind": kind, "total": total, "images": n_img,
            "videos": n_vid, "labels": labels, "raw_gb": raw_gb}


results = {}
for name, path in ARCHIVES.items():
    if Path(path).exists():
        results[name] = inspect(name, path)
    else:
        print(f"SKIP {name}: {path} not found")
        print()

## 5 · Summary

In [ ]:
NEXT = {
    "IMAGES":     "READY        -> unzip to /content/data/, build manifest, train",
    "VIDEOS":     "EXTRACT      -> MTCNN pass needed before training",
    "NESTED_ZIP": "UNZIP ONCE   -> then re-run this notebook on the inner zips",
    "MIXED":      "INSPECT      -> read the sample paths above by hand",
}

print("=" * 72)
print("SUMMARY")
print("=" * 72)

for name, r in results.items():
    if not r:
        continue
    print()
    print(f"  {name}")
    print(f"      contents : {r['kind']}  ({r['total']:,} files, {r['raw_gb']:.1f} GB unzipped)")
    print(f"      labels   : {r['labels'].get(0,0):,} real / "
          f"{r['labels'].get(1,0):,} fake / {r['labels'].get(-1,0):,} unresolved")
    print(f"      next     : {NEXT[r['kind']]}")

kinds = {r["kind"] for r in results.values() if r}
total_gb = sum(r["raw_gb"] for r in results.values() if r)

print()
print("=" * 72)
if kinds == {"IMAGES"}:
    print("All archives are face crops. You can go straight to training.")
    print(f"Unzipped they need ~{total_gb:.1f} GB on /content.")
elif "VIDEOS" in kinds:
    print("At least one archive is raw video.")
    print("build_manifest.py only scans image files, so it finds nothing there.")
    print("A one-time MTCNN extraction pass is needed first - budget several")
    print("hours on a T4. Write the crops back to Drive so a disconnect never")
    print("costs you that work twice.")
else:
    print("Mixed or nested. Read the per-archive detail above.")
print("=" * 72)

---

## How to read the result

**`IMAGES`** — your zips are already face crops. Unzip to `/content/data/`
(local disk, never train off mounted Drive) and run
`python -m data.build_manifest --dry-run`.

**`VIDEOS`** — the videos still need converting into face crops before training.
That is a one-time MTCNN pass over every video, and it is the expensive step:
budget several hours on a T4 for ~11,500 videos. Save the crops back to Drive
afterwards so you never repeat it.

**Unresolved labels** — `build_manifest.py` decides real vs fake purely from
words in the file path. A high unresolved count means its token lists don't match
your folder names. Fix that *before* running the full scan.

**Median frames per folder below 16** — `DATA.clip_len = 16` asks for 16 frames
per clip. With fewer, `ClipDataset._sample_frames` clamps and repeats the last
frame, so clips get padded with duplicates. Lower `clip_len` instead.

### If disk is tight

The T4 runtime has ~78 GB free, so both archives fit easily. If you later add
DF40, unzip one at a time and delete each zip after extracting:

```bash
cp /content/drive/MyDrive/FF++.zip /content/
unzip -q /content/FF++.zip -d /content/data/
rm /content/FF++.zip
```